# EDA on NASAs eclipse catalogs

* **Purpose:** to understand the two raw files well enough to determine which columns the dashboard requires, and to produce a cleaned `Parquet` file that the backend can read without needing to perform any further cleaning.

>The raw data in `backend/data/raw/` is considered my `SSOT`. 

>All output and clean data is stored in `backend/data/processed/` which will the data thats going to be used in the `serving layer`.


### Pathing
* Make sure pathing is correct and define paths with ROOT_DIR

#### **Output:**
````
solar.csv       -> exists
lunar.csv       -> exists


In [19]:
from pathlib import Path
import pandas as pd

# Hitta vart jag står
current_dir = Path.cwd()

# Hitta root, om current_dir är i EDA/ så är root parent.
# Om VScode redan startar i root sätter jag root = current_dir
if current_dir.name == "EDA":
    ROOT_DIR = current_dir.parent
else:
    ROOT_DIR = current_dir

# Definiera paths utifrån ROOT_DIR
RAW = ROOT_DIR / "backend" / "data" / "raw"
PROCESSED = ROOT_DIR / "backend" / "data" / "processed"

SOLAR_CSV = RAW / "solar.csv"
LUNAR_CSV = RAW / "lunar.csv"

# test
for f in (SOLAR_CSV, LUNAR_CSV):
    print(f"{f.name:15} -> {'exists' if f.exists() else 'missing'}")

solar.csv       -> exists
lunar.csv       -> exists


# Solar dataset

## 1) Finding the shape of my data and letting pandas infer its own schema

#### **Output:**

```
Solar: (11898, 15)
Lunar: (12064, 16)

In [20]:
solar_str = pd.read_csv(SOLAR_CSV, dtype=str)
lunar_str = pd.read_csv(LUNAR_CSV, dtype=str)

solar = pd.read_csv(SOLAR_CSV)
lunar = pd.read_csv(LUNAR_CSV)

print("Solar:", solar.shape)
print("Lunar:", lunar.shape)

Solar: (11898, 15)
Lunar: (12064, 16)


## 2) Looking at the actual rows in my data

* `.head()` and `.tail()` will show the edges and `.sample()` shows the middle.
* `random_state` fixes the sample so my notebook produces the same rows every run to not confuse others people cloning the repo and getting different output compared to the output under.

    * The point of this is to see real values for yourself before doing any statistics on them.

In [21]:
display(solar.head())
display(solar.tail())
display(solar.sample(5, random_state=69))

,Catalog Number,Calendar Date,Eclipse Time,Delta T (s),Lunation Number,Saros Number,Eclipse Type,Gamma,Eclipse Magnitude,Latitude,Longitude,Sun Altitude,Sun Azimuth,Path Width (km),Central Duration
0,1,-1999 June 12,03:14:51,46438,-49456,5,T,-0.2701,1.0733,6.0N,33.3W,74,344,247,06m37s
1,2,-1999 December 5,23:45:23,46426,-49450,10,A,-0.2317,0.9382,32.9S,10.8E,76,21,236,06m44s
2,3,-1998 June 1,18:09:16,46415,-49444,15,T,0.4994,1.0284,46.2N,83.4E,60,151,111,02m15s
3,4,-1998 November 25,05:57:03,46403,-49438,20,A,-0.9045,0.9806,67.8S,143.8W,25,74,162,01m14s
4,5,-1997 April 22,13:19:56,46393,-49433,-13,P,-1.4670,0.1611,60.6S,106.4W,0,281,NaN,NaN


,Catalog Number,Calendar Date,Eclipse Time,Delta T (s),Lunation Number,Saros Number,Eclipse Type,Gamma,Eclipse Magnitude,Latitude,Longitude,Sun Altitude,Sun Azimuth,Path Width (km),Central Duration
11893,11894,2998 December 10,03:18:31,4414,12355,187,P,1.2838,0.4773,67.2N,145.0E,0,179,NaN,NaN
11894,11895,2999 May 6,23:23:57,4417,12360,154,T,0.8388,1.0566,71.5N,177.3E,33,146,345,03m25s
11895,11896,2999 October 30,09:34:33,4420,12366,159,A-,-1.0023,0.9586,70.9S,84.7W,0,137,-,-
11896,11897,3000 April 26,14:18:06,4424,12372,164,T,0.1310,1.0222,21.1N,18.4W,82,166,76,02m11s
11897,11898,3000 October 19,16:10:16,4428,12378,169,H,-0.2303,1.0049,23.1S,51.6W,77,16,17,00m29s


,Catalog Number,Calendar Date,Eclipse Time,Delta T (s),Lunation Number,Saros Number,Eclipse Type,Gamma,Eclipse Magnitude,Latitude,Longitude,Sun Altitude,Sun Azimuth,Path Width (km),Central Duration
4809,4810,15 March 9,05:44:12,10382,-24549,59,T,-0.6707,1.0537,41.3S,165.7E,48,321,236,03m49s
4165,4166,-252 January 21,00:55:45,13379,-27853,56,T,0.3453,1.0287,0.9S,136.3W,70,175,104,02m57s
1058,1059,-1563 January 24,14:27:55,36442,-44068,35,P,1.1965,0.6282,63.5N,83.6E,0,148,NaN,NaN
61,62,-1974 August 3,11:35:27,45828,-49145,4,H,0.5677,1.0057,55.0N,147.6W,55,205,24,00m27s
5364,5365,247 March 24,10:08:36,8152,-21679,72,T,0.2796,1.0538,16.6N,58.4E,74,161,185,04m50s


## 3) Column overview
* `.info()` gives four things per column which are: 
    * name
    * non-null count
    * dtype
    * memory usage

* **Two things to read out of it:** Any column where the non null count is below `11898` has missing values and any column stored as `str` that looks numeric is a column pandas cound *not* parse.

#### **Output for .info():**
```
<class 'pandas.DataFrame'>
RangeIndex: 11898 entries, 0 to 11897
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Catalog Number     11898 non-null  int64  
 1   Calendar Date      11898 non-null  str    
 2   Eclipse Time       11898 non-null  str    
 3   Delta T (s)        11898 non-null  int64  
 4   Lunation Number    11898 non-null  int64  
 5   Saros Number       11898 non-null  int64  
 6   Eclipse Type       11898 non-null  str    
 7   Gamma              11898 non-null  float64
 8   Eclipse Magnitude  11898 non-null  float64
 9   Latitude           11898 non-null  str    
 10  Longitude          11898 non-null  str    
 11  Sun Altitude       11898 non-null  int64  
 12  Sun Azimuth        11898 non-null  int64  
 13  Path Width (km)    7698 non-null   str    
 14  Central Duration   7698 non-null   str    
dtypes: float64(2), int64(6), str(7)
memory usage: 1.8 MB
```

In [22]:
solar.info()

<class 'pandas.DataFrame'>
RangeIndex: 11898 entries, 0 to 11897
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Catalog Number     11898 non-null  int64  
 1   Calendar Date      11898 non-null  str    
 2   Eclipse Time       11898 non-null  str    
 3   Delta T (s)        11898 non-null  int64  
 4   Lunation Number    11898 non-null  int64  
 5   Saros Number       11898 non-null  int64  
 6   Eclipse Type       11898 non-null  str    
 7   Gamma              11898 non-null  float64
 8   Eclipse Magnitude  11898 non-null  float64
 9   Latitude           11898 non-null  str    
 10  Longitude          11898 non-null  str    
 11  Sun Altitude       11898 non-null  int64  
 12  Sun Azimuth        11898 non-null  int64  
 13  Path Width (km)    7698 non-null   str    
 14  Central Duration   7698 non-null   str    
dtypes: float64(2), int64(6), str(7)
memory usage: 1.8 MB


## 4) Summary statistics
* `.describe()` covers all the numeric columns such as:
    * count
    * mean
    * std
    * min
    * the three quartiles
    * max

* `include="all"` adds the text cols, which are described in a different way:
    * count
    * unique
    * top
    * freq

* Read min and max FIRST, its here where the impossible values SHOULD show up.

---

#### **Output for describe():**

| Quartiles | Catalog Number | Delta T (s) | Lunation Number | Saros Number | Gamma | Eclipse Magnitude | Sun Altitude | Sun Azimuth | 
|:--|:--|:--|:--|:--|:--|:--|:--|:--|
| count| 11898.000000| 11898.000000| 11898.000000| 11898.000000| 11898.000000| 11898.000000 | 11898.000000 | 11898.000000 |
| mean| 5949.500000| 12142.172802| -18546.959321| 87.483190| -0.002469| 0.812748| 36.505295| 180.264330|
| std| 3434.801086| 13583.402888| 17906.572982| 48.380284| 0.900860| 0.300398| 32.417350| 110.745408|
| min| 1.000000| -6.000000| -49456.000000| -13.000000| -1.569000| 0.000000| 0.000000| 0.000000|
| 25%| 2975.250000| 970.250000| -33954.750000| 47.000000| -0.786575| 0.675925| 0.000000| 89.000000|
| 50%| 5949.500000| 5636.500000| -18495.000000| 87.000000| -0.003850| 0.950600| 38.000000| 180.000000|
| 75%| 8923.750000| 20943.500000| -3039.250000| 128.000000| 0.776900| 1.018400| 66.000000| 272.000000|
| max| 11898.000000| 46438.000000| 12378.000000| 190.000000| 1.570600| 1.081300| 90.000000| 360.000000|

---
### **Output for decsribe(include="all"):**

| Quartiles| Catalog Number| Calendar Date| Eclipse Time| Delta T (s)| Lunation Number| Saros Number| Eclipse Type| Gamma| Eclipse Magnitude| Latitude| Longitude| Sun Altitude| Sun Azimuth | Path Width (km) | Central Duration|
|:--|:--|:--|:--|:--|:--|:--|:--|:--|:--|:--|:--|:--|:--|:--|:--|
| count| 11898.000000| 11898| 11898| 11898.000000| 11898.000000| 11898.000000| 11898| 11898.000000| 11898.000000| 11898| 11898| 11898.000000| 11898.000000| 7698| 7698|
| unique| NaN| 11898| 11154| NaN| NaN| NaN| 19| NaN| NaN| 1654| 3448| NaN| NaN| 754| 679|
| top| NaN| -1999 June 12| 00:40:11| NaN| NaN| NaN| P| NaN| NaN| 61.0S| 151.7W| NaN| NaN| -| -|
| freq| NaN| 1| 4| NaN| NaN| NaN| 3875| NaN| NaN| 69| 10| NaN| NaN| 181| 94|
| mean|	5949.500000|	NaN|	NaN|	12142.172802|	-18546.959321|	87.483190|	NaN	|-0.002469|	0.812748|	NaN	|NaN	|36.505295|	180.264330|	NaN| NaN|
| std|	3434.801086|	NaN|	NaN|	13583.402888|	17906.572982|	48.380284|	NaN|	0.900860|	0.300398|	NaN	|NaN	|32.417350|	110.745408|	NaN| NaN|
| min| 1.000000| NaN| NaN| -6.000000| -49456.000000| -13.000000| NaN| -1.569000| 0.000000| NaN| NaN| 0.000000| 0.000000| NaN| NaN|
| 25%| 2975.250000| NaN| NaN| 970.250000| -33954.750000| 47.000000| NaN| -0.786575| 0.675925| NaN| NaN| 0.000000| 89.000000| NaN| NaN|
| 50%| 5949.500000| NaN| NaN| 5636.500000| -18495.000000| 87.000000| NaN| -0.003850| 0.950600| NaN| NaN| 38.000000| 180.000000| NaN| NaN|
| 75%| 8923.750000| NaN| NaN| 20943.500000| -3039.250000| 128.000000| NaN| 0.776900| 1.018400| NaN| NaN| 66.000000| 272.000000| NaN| NaN|
| max| 11898.000000| NaN| NaN| 46438.000000| 12378.000000| 190.000000| NaN| 1.570600| 1.081300| NaN| NaN| 90.000000| 360.000000| NaN| NaN|

In [23]:
solar.describe()

,Catalog Number,Delta T (s),Lunation Number,Saros Number,Gamma,Eclipse Magnitude,Sun Altitude,Sun Azimuth
count,11898.000000,11898.000000,11898.000000,11898.000000,11898.000000,11898.000000,11898.000000,11898.000000
mean,5949.500000,12142.172802,-18546.959321,87.483190,-0.002469,0.812748,36.505295,180.264330
std,3434.801086,13583.402888,17906.572982,48.380284,0.900860,0.300398,32.417350,110.745408
min,1.000000,-6.000000,-49456.000000,-13.000000,-1.569000,0.000000,0.000000,0.000000
25%,2975.250000,970.250000,-33954.750000,47.000000,-0.786575,0.675925,0.000000,89.000000
50%,5949.500000,5636.500000,-18495.000000,87.000000,-0.003850,0.950600,38.000000,180.000000
75%,8923.750000,20943.500000,-3039.250000,128.000000,0.776900,1.018400,66.000000,272.000000
max,11898.000000,46438.000000,12378.000000,190.000000,1.570600,1.081300,90.000000,360.000000


In [24]:
# I pandas 3.x.x har txt cols dtype "str" och inte "object"
# Den gamla vanan med att använda describe(include="object") ger därför en helt en tom tabell och kommer ej funka
solar.describe(include="all")

,Catalog Number,Calendar Date,Eclipse Time,Delta T (s),Lunation Number,Saros Number,Eclipse Type,Gamma,Eclipse Magnitude,Latitude,Longitude,Sun Altitude,Sun Azimuth,Path Width (km),Central Duration
count,11898.000000,11898,11898,11898.000000,11898.000000,11898.000000,11898,11898.000000,11898.000000,11898,11898,11898.000000,11898.000000,7698,7698
unique,NaN,11898,11154,NaN,NaN,NaN,19,NaN,NaN,1654,3448,NaN,NaN,754,679
top,NaN,-1999 June 12,00:40:11,NaN,NaN,NaN,P,NaN,NaN,61.0S,151.7W,NaN,NaN,-,-
freq,NaN,1,4,NaN,NaN,NaN,3875,NaN,NaN,69,10,NaN,NaN,181,94
mean,5949.500000,NaN,NaN,12142.172802,-18546.959321,87.483190,NaN,-0.002469,0.812748,NaN,NaN,36.505295,180.264330,NaN,NaN
std,3434.801086,NaN,NaN,13583.402888,17906.572982,48.380284,NaN,0.900860,0.300398,NaN,NaN,32.417350,110.745408,NaN,NaN
min,1.000000,NaN,NaN,-6.000000,-49456.000000,-13.000000,NaN,-1.569000,0.000000,NaN,NaN,0.000000,0.000000,NaN,NaN
25%,2975.250000,NaN,NaN,970.250000,-33954.750000,47.000000,NaN,-0.786575,0.675925,NaN,NaN,0.000000,89.000000,NaN,NaN
50%,5949.500000,NaN,NaN,5636.500000,-18495.000000,87.000000,NaN,-0.003850,0.950600,NaN,NaN,38.000000,180.000000,NaN,NaN
75%,8923.750000,NaN,NaN,20943.500000,-3039.250000,128.000000,NaN,0.776900,1.018400,NaN,NaN,66.000000,272.000000,NaN,NaN


## 5) Missing values
* Count and percentage for each column / per column. Percentage is that which tells you wether a gap matters or not.
* When two cols are missing the same exact number of rows i should check wether they are the same rows or not. A shared gap usually means a rule in the data and not an error.


#### **Output:**


| column | missing | percent|
|:--|:--|:--|
|Path Width (km) | 4200| 35.3|
|Central Duration | 4200| 35.3|

`Are the exact same rows missing in both columns? True`

|Eclipse | Type |
|:--|:--|
|P     | 3875|
|Pb    |  163|
|Pe    |  162|
|Name: count | dtype: int64 | 

In [25]:
display(
    pd.DataFrame(
        {
            "missing": solar.isna().sum(),
            "percent": (solar.isna().mean() * 100).round(2),
        }
    ).query("missing > 0")
)

# Är det 4200 raderna UTAN path width exakt samma rader som central duration?
# Jämför båda raderna med varandra för att få ut True/False bool
same_rows = (solar["Path Width (km)"].isna() == solar["Central Duration"].isna()).all()
print("Are the exact same rows missing in both columns?", same_rows)
print("\n")
# Vilken förmörkelse typ är det som saknas?
print(solar.loc[solar["Path Width (km)"].isna(), "Eclipse Type"].value_counts())

,missing,percent
Path Width (km),4200,35.3
Central Duration,4200,35.3


Are the exact same rows missing in both columns? True


Eclipse Type
P     3875
Pb     163
Pe     162
Name: count, dtype: int64


## 6) Duplicate count / duplicates
* Whole rows first then column that is supposed to be unique


#### **Output:**

> Duplicate rows in solar data: 0

> duplicate catalog numbers: 0

In [26]:
print("Duplicate rows in solar data:", solar.duplicated().sum())
print("\n\n")
print("duplicate catalog numbers:", solar["Catalog Number"].duplicated().sum())

Duplicate rows in solar data: 0



duplicate catalog numbers: 0


## 7) Cardinality and Value counts.
* `.nunique()` puts the categorical columns at the top and sorts ascending.
    * A col with few disctinct values is a candidate for filter functions in the dashboard
    * A col with only one distinct value gives / carries barely any information at all.


#### **Output:**

```
Eclipse Type            19
Sun Altitude            91
Saros Number           204
Sun Azimuth            361
Central Duration       679
Path Width (km)        754
Latitude              1654
Longitude             3448
Eclipse Magnitude     4889
Delta T (s)           9190
Gamma                 9814
Eclipse Time         11154
Lunation Number      11898
Catalog Number       11898
Calendar Date        11898
dtype: int64
```

```
Eclipse Type
P     3875
A     3755
T     3049
H      502
Pb     163
Pe     162
Tm      72
Am      72
An      36
A+      34
A-      34
H3      26
As      25
H2      24
Hm      17
T-      17
Tn      14
Ts      12
T+       9
Name: count, dtype: int64
```


In [27]:
display(solar.nunique().sort_values())
print("===================")
display(solar["Eclipse Type"].value_counts())

Eclipse Type            19
Sun Altitude            91
Saros Number           204
Sun Azimuth            361
Central Duration       679
Path Width (km)        754
Latitude              1654
Longitude             3448
Eclipse Magnitude     4889
Delta T (s)           9190
Gamma                 9814
Eclipse Time         11154
Lunation Number      11898
Catalog Number       11898
Calendar Date        11898
dtype: int64

Eclipse Type
P     3875
A     3755
T     3049
H      502
Pb     163
Pe     162
Tm      72
Am      72
An      36
A+      34
A-      34
H3      26
As      25
H2      24
Hm      17
T-      17
Tn      14
Ts      12
T+       9
Name: count, dtype: int64

# Lunar Eclipses.

* Since I intend to ask questions that are IDENTICAL to the solar EDA above but for my Lunar dataset in  `backend/data/raw/lunar.csv` I will run the same steps but in two code cells instead of six rather than typing everything twice, adhearing to the `DRY` principles.

In [28]:
# Hoppar över .tail() och .sample() då jag testat dessa statements och ej vill upprepa mig allt för mycket
display(lunar.head())

,Catalog Number,Calendar Date,Eclipse Time,Delta T (s),Lunation Number,Saros Number,Eclipse Type,Quincena Solar Eclipse,Gamma,Penumbral Magnitude,Umbral Magnitude,Latitude,Longitude,Penumbral Eclipse Duration (m),Partial Eclipse Duration (m),Total Eclipse Duration (m)
0,1,-1999 June 26,14:13:28,46437,-49456,17,N,t-,-1.0981,0.8791,-0.1922,24S,22W,268.8,-,-
1,2,-1999 November 21,20:23:49,46427,-49451,-16,N,-a,-1.1155,0.8143,-0.1921,15N,98W,233.4,-,-
2,3,-1998 May 17,05:47:36,46416,-49445,-11,P,-t,0.8988,1.2105,0.2069,13S,89E,281.7,102.7,-
3,4,-1998 November 11,05:15:58,46404,-49439,-6,P,-a,-0.4644,2.0382,0.9740,12N,113E,343.4,200.8,-
4,5,-1997 May 6,18:57:01,46392,-49433,-1,T+,pp,0.1003,2.6513,1.6963,11S,92W,322.8,213.5,98.2


In [29]:
lunar.info()

<class 'pandas.DataFrame'>
RangeIndex: 12064 entries, 0 to 12063
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Catalog Number                  12064 non-null  int64  
 1   Calendar Date                   12064 non-null  str    
 2   Eclipse Time                    12064 non-null  str    
 3   Delta T (s)                     12064 non-null  int64  
 4   Lunation Number                 12064 non-null  int64  
 5   Saros Number                    12064 non-null  int64  
 6   Eclipse Type                    12064 non-null  str    
 7   Quincena Solar Eclipse          12064 non-null  str    
 8   Gamma                           12064 non-null  float64
 9   Penumbral Magnitude             12064 non-null  float64
 10  Umbral Magnitude                12064 non-null  float64
 11  Latitude                        12064 non-null  str    
 12  Longitude                       12064 non-n

In [30]:
display(lunar.describe())

,Catalog Number,Delta T (s),Lunation Number,Saros Number,Gamma,Penumbral Magnitude,Umbral Magnitude,Penumbral Eclipse Duration (m)
count,12064.000000,12064.000000,12064.000000,12064.000000,12064.000000,12064.000000,12064.000000,12064.000000
mean,6032.500000,12116.476044,-18531.392822,80.505056,0.002490,1.418656,0.400175,269.971941
std,3482.721158,13584.785584,17887.218741,48.416007,0.910505,0.832263,0.832954,79.945444
min,1.000000,-6.000000,-49456.000000,-20.000000,-1.582700,0.000400,-1.095800,5.200000
25%,3016.750000,962.000000,-33923.250000,40.000000,-0.788825,0.684425,-0.334000,223.100000
50%,6032.500000,5597.000000,-18445.500000,80.000000,0.001750,1.417450,0.400450,295.000000
75%,9048.250000,20901.500000,-3067.750000,121.000000,0.791725,2.136900,1.117925,327.800000
max,12064.000000,46437.000000,12378.000000,183.000000,1.579100,2.908900,1.882100,379.500000


### I will compress step 5, 6 and 7 into one cell for Lunar.

* What this cell will contain:
    * Missing values
    * Duplicate count
    * Cardinality(distinct count, using `nunique()` - Number of unique)

In [31]:
# Missing values, steg 5 ovanför för Solar
display(
    pd.DataFrame(
        {
            "missing": lunar.isna().sum(),
            "percent": (lunar.isna().mean() * 100).round(1),
        }
    ).query("missing > 0")
)
print("==========")
# Duplicate count - Steg 6 ovan för solar
print("Duplicate rows for Lunar data:", lunar.duplicated().sum())
print("==========")
print("Duplicate catalog numbers for Lunar data:", lunar["Catalog Number"].duplicated().sum())
print("==========")
# Cardinality(distinct count, nummer av unika värden i cols) - Steg 7 ovan för Solar
display(lunar.nunique().sort_values())
print("==========")
display(lunar["Eclipse Type"].value_counts())



,missing,percent


Duplicate rows for Lunar data: 0
Duplicate catalog numbers for Lunar data: 0


Eclipse Type                          8
Quincena Solar Eclipse               11
Latitude                             52
Saros Number                        204
Longitude                           362
Total Eclipse Duration (m)          809
Partial Eclipse Duration (m)       1808
Penumbral Eclipse Duration (m)     2965
Delta T (s)                        9279
Umbral Magnitude                   9859
Penumbral Magnitude                9899
Gamma                              9974
Eclipse Time                      11198
Lunation Number                   12064
Calendar Date                     12064
Catalog Number                    12064
dtype: int64

Eclipse Type
P     4207
N     4020
T     1405
T+    1042
T-    1032
Nx     141
Ne     115
Nb     102
Name: count, dtype: int64

## Interesting findings in the Lunar data that needs to be checked deeper.
When running this to find missing values it shows no missing values which is odd.
```python
display(
    pd.DataFrame(
        {
            "missing": lunar.isna().sum(),
            "percent": (lunar.isna().mean() * 100).round(1),
        }
    ).query("missing > 0")
)
```
* Running `display(lunar.head())` above it showed `-` as values on these two columns: 
    * Partial Eclipse Duration (m)	
    * Total Eclipse Duration (m)

* While running `lunar.info()` these three columns shows up as these datatypes:
    * Penumbral Eclipse Duration (m)  12064 non-null  float64
    * Partial Eclipse Duration (m)    12064 non-null  str    
    * Total Eclipse Duration (m)      12064 non-null  str 

* After some further investigation it appears Nasa has written a dash/hyphen `-` where the duration doesnt exist. For `pandas` a hyphen is just a normal `str`, the value is present so `isna()` returns 0. 

If I take a look at the output for Eclipse Types:
```
Eclipse Type
P     4207
N     4020
T     1405
T+    1042
T-    1032
Nx     141
Ne     115
Nb     102
Name: count, dtype: int64
```
This completely explains the hidden missing values! 
The dataset contains 12 064 lunar eclipses in total, but they are split into several different categories:
* **Total eclipses (T, T+, T-):** 1405 + 1042 + 1032 = 3479
* **Partial eclipses (P):** 4207
* **Penumbral eclipses (N, Nx, Ne, Nb):** 4020 + 141 + 115 + 102 = 4378

This would mean that a lunar eclipse can only have a `Total Eclipse Duration` if it *actually* reaches the total phase. If I take the total number of eclipses **12 064** and then subtract the ones that are actually total eclipses **3479**, then what is left is *exactly* **8585** partial and penumbral eclipses. 

These **8585** eclipses does *not* have a total duration and instead of leaving the rows blank (which Pandas in my case would read as `NaN`)  NASA must have decided to populated them with a hyphen `-`. This would perfectly explains why the column is cast as a `str` and why `isna().sum()` returns a 0.

## Type check for Lunar dataset.
* Lunar reported 0 missing values as mentioned in cells above, now I will see if my thoughts about NASA populating rows with a `'-'` where the duration does not exist.
    * The solar dataset left the same fields `empty` which is why pandas read them as missing. Same rule, two notations and only *ONE* of them is visible.

#### **Output:**
```text
Partial Eclipse Duration (m)    4378
Total Eclipse Duration (m)      8585
dtype: int64
==========
Path Width (km)     181
Central Duration     94
dtype: int64
```
---
**Conclusion:**
This output perfectly explains the "hidden missing values". The missing data does not mean eclipses are lost, it means those specific eclipses never reached that phase.

* There are **12 064** lunar eclipses in total.
* Only **3479** of them are *total* eclipses **(Types: T, T+, T-)**
* **12 064 - 3479 = 8585**.

Exactly **8585** eclipses *never* reached a *total* phase, which perfectly matches the 8585 hyphens (`'-'`) found in the `Total Eclipse Duration (m)` column.

In [32]:
# Räknar bindestreck per col, numeric cols ska alltid ge false här, 
# så det som bör visa sig här är text som ska betyda 'inget värde'

display((lunar == "-").sum().loc[lambda s: s > 0])
print("==========")
display((solar == "-").sum().loc[lambda s: s > 0])

Partial Eclipse Duration (m)    4378
Total Eclipse Duration (m)      8585
dtype: int64

Path Width (km)     181
Central Duration     94
dtype: int64

# Column decisions + cleaning of data 

In [33]:
# Funktion för att omvandla koordinater till floats
# Latitude koordinater för S är NEGATIVA
# Longitude koordinater för W är NEGATIVA
def parse_coordinates(series: pd.Series, negative_for: str) -> pd.Series:
    """Turn coordinates '6.0N', '33.3W' into a float."""
    # Sista tecknet är väderstrecket och resten är talet
    directions = series.str[-1]
    # Sätta float64 som EXPLICIT datatype, Lunar skriver koordinater UTAN decimaler...
    # to_numeric ger mig int64 för lunar och float64 för solars koordinater
    # Schemat MÅSTE vara IDENTISKT för dashboarden.
    value = pd.to_numeric(series.str[:-1], errors="coerce").astype("float64")
    return value.where(directions != negative_for, -value)

# Funktion för att parsa åren
def parse_years(series: pd.Series) -> pd.Series:
    """Take the year out of '-1999 June 27'"""
    # Året kommer behållas som int och inte som datetime, negativa är år f.kr(före kristus)
    # och datetime kan omöjligen representera något från den tiden
    return pd.to_numeric(series.str.split().str[0], errors="coerce").astype("int64")


# Funktion för att bygga min 'core' med samma form som båda datasettens catalogs
def build_core(df: pd.DataFrame, magnitude_columns: str, body: str) -> pd.DataFrame:
    """The typed 'core' that the backend will read. Exact same shape for both catalogs as in the raw datasets."""
    return pd.DataFrame(
        {
            "catalog_number": df["Catalog Number"].astype("int64"),
            "calendar_date": df["Calendar Date"], # Behåller det som TEXT pga BCE och negativa år innan vår tidsräkning började
            "year": parse_years(df["Calendar Date"]),
            "eclipse_type": df["Eclipse Type"].str[0], # Första tecknet är TYPEN
            "saros_number": df["Saros Number"].astype("int64"),
            "latitude": parse_coordinates(df["Latitude"], negative_for="S"),
            "longitude": parse_coordinates(df["Longitude"], negative_for="W"),
            "magnitude": df[magnitude_columns].astype("float64"),
            "body": body,
        }

    )

solar_clean = build_core(solar, magnitude_columns="Eclipse Magnitude", body="solar")
lunar_clean = build_core(lunar, magnitude_columns="Umbral Magnitude", body="lunar")

display(solar_clean.head(10))
display(solar_clean.dtypes)
print(solar_clean["eclipse_type"].value_counts().to_dict())
print(lunar_clean["eclipse_type"].value_counts().to_dict())


,catalog_number,calendar_date,year,eclipse_type,saros_number,latitude,longitude,magnitude,body
0,1,-1999 June 12,-1999,T,5,6.0,-33.3,1.0733,solar
1,2,-1999 December 5,-1999,A,10,-32.9,10.8,0.9382,solar
2,3,-1998 June 1,-1998,T,15,46.2,83.4,1.0284,solar
3,4,-1998 November 25,-1998,A,20,-67.8,-143.8,0.9806,solar
4,5,-1997 April 22,-1997,P,-13,-60.6,-106.4,0.1611,solar
5,6,-1997 May 22,-1997,P,25,61.7,-151.7,0.4035,solar
6,7,-1997 October 16,-1997,P,-8,60.6,-22.7,0.6954,solar
7,8,-1997 November 14,-1997,P,30,-61.5,-27.7,0.0377,solar
8,9,-1996 April 10,-1996,A,-3,-38.2,-167.2,0.9464,solar
9,10,-1996 October 4,-1996,T,2,28.8,38.6,1.0257,solar


catalog_number      int64
calendar_date         str
year                int64
eclipse_type          str
saros_number        int64
latitude          float64
longitude         float64
magnitude         float64
body                  str
dtype: object

{'P': 4200, 'A': 3956, 'T': 3173, 'H': 569}
{'N': 4378, 'P': 4207, 'T': 3479}


In [34]:
# Sanity check för att datatypes stämmer överens
schema = pd.DataFrame({"solar": solar_clean.dtypes, "lunar": lunar_clean.dtypes})
assert solar_clean.dtypes.equals(lunar_clean.dtypes), f"Schemas differ:\n{schema}"
display(schema)

,solar,lunar
catalog_number,int64,int64
calendar_date,str,str
year,int64,int64
eclipse_type,str,str
saros_number,int64,int64
latitude,float64,float64
longitude,float64,float64
magnitude,float64,float64
body,str,str


## Write the clean and processed data to backend/data/processed 

- Write the cleaned data as clean .parquet files to optimize for storage since I want to spend as little as possible for upcoming steps when deploying it to Azure.

**Output:**
All columns are the correct datatype when reading back from my cleaned data



In [35]:
PROCESSED.mkdir(parents=True, exist_ok=True)
solar_clean.to_parquet(PROCESSED / "solar.parquet", index=False)
lunar_clean.to_parquet(PROCESSED / "lunar.parquet", index=False)

## Benchmarking to compare pandas reading time for both CSV vs PARQUET.
- Reason: because its fun and if I am optimizing for storage costs, I want to try to optimize for speed as well.


**OUTPUT AFTER TEST:**
```text
solar   - 
CSV     -    1014.0 KB     26.7 ms
Parquet -     313.4 KB      3.9 ms
lunar   - 
CSV     -    1089.9 KB     27.7 ms
Parquet -     312.0 KB      3.5 ms
```


In [36]:
import time

for name in ("solar", "lunar"):
    csv_path = RAW / f"{name}.csv"
    parquet_path = PROCESSED / f"{name}.parquet"

    start = time.perf_counter()
    pd.read_csv(csv_path)
    csv_ms = (time.perf_counter() - start) * 1000

    start = time.perf_counter()
    pd.read_parquet(parquet_path)
    parquet_ms = (time.perf_counter() - start) * 1000

    print(f"{name}   - ")
    print(f"CSV     -   {csv_path.stat().st_size / 1024:7.1f} KB   {csv_ms:6.1f} ms")
    print(f"Parquet -   {parquet_path.stat().st_size / 1024:7.1f} KB   {parquet_ms:6.1f} ms")

solar   - 
CSV     -    1014.0 KB     28.5 ms
Parquet -     313.4 KB     19.2 ms
lunar   - 
CSV     -    1089.9 KB     31.6 ms
Parquet -     312.1 KB     17.1 ms
